# YouTube Dataset - Load and Extract Random Short/Long Video with Parent Video Analysis

In [14]:
import pandas as pd
import numpy as np
import json
import random
import re
from pathlib import Path
from urllib.parse import urlparse, parse_qs

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

print("Libraries loaded successfully")

Libraries loaded successfully


In [15]:
# Load YouTube videos metadata
youtube_data_path = Path('../Data/YouTube/videos_metadata.csv')

if youtube_data_path.exists():
    df = pd.read_csv(youtube_data_path)
    print(f"Dataset loaded successfully!")
    print(f"Shape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
    print(f"\nFirst few rows:")
    print(df.head())
else:
    print(f"File not found at {youtube_data_path}")
    print(f"Current working directory: {Path.cwd()}")
    print(f"Available parent directory contents:")
    print(list(Path('../').glob('*')))

Dataset loaded successfully!
Shape: (12839, 67)

Columns: ['videoId', 'publishedAt', 'channelId', 'channelTitle', 'title', 'description', 'tags', 'categoryId', 'liveBroadcastContent', 'defaultLanguage', 'defaultAudioLanguage', 'localized_title', 'localized_description', 'thumbnail_default_url', 'thumbnail_default_width', 'thumbnail_default_height', 'thumbnail_medium_url', 'thumbnail_medium_width', 'thumbnail_medium_height', 'thumbnail_high_url', 'thumbnail_high_width', 'thumbnail_high_height', 'thumbnail_standard_url', 'thumbnail_standard_width', 'thumbnail_standard_height', 'thumbnail_maxres_url', 'thumbnail_maxres_width', 'thumbnail_maxres_height', 'duration', 'dimension', 'definition', 'caption', 'licensedContent', 'projection', 'hasCustomThumbnail', 'regionRestriction_allowed', 'regionRestriction_blocked', 'uploadStatus', 'failureReason', 'rejectionReason', 'privacyStatus', 'publishAt', 'license', 'embeddable', 'publicStatsViewable', 'madeForKids', 'selfDeclaredMadeForKids', 'conta

C:\Users\alire\AppData\Local\Temp\ipykernel_22944\839863283.py:5: DtypeWarning: Columns (35,62,65) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(youtube_data_path)


In [16]:
# Helper function to parse ISO 8601 duration format
def parse_duration(duration_str):
    """Parse ISO 8601 duration format (PT1H2M3S) to total seconds"""
    if pd.isna(duration_str):
        return None
    
    pattern = r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?'
    match = re.match(pattern, str(duration_str))
    
    if match:
        hours = int(match.group(1)) if match.group(1) else 0
        minutes = int(match.group(2)) if match.group(2) else 0
        seconds = int(match.group(3)) if match.group(3) else 0
        return hours * 3600 + minutes * 60 + seconds
    return None

# Parse duration and check for short videos
if 'duration' in df.columns:
    df['duration_seconds'] = df['duration'].apply(parse_duration)
    print(f"Duration parsed successfully!")
    print(f"Duration range: {df['duration_seconds'].min()} to {df['duration_seconds'].max()} seconds")
    print(f"\nDuration statistics:")
    print(df['duration_seconds'].describe())
else:
    print("No duration column found")

Duration parsed successfully!
Duration range: 2.0 to 42901.0 seconds

Duration statistics:
count    12829.000000
mean       675.949411
std        955.704322
min          2.000000
25%         55.000000
50%        357.000000
75%        979.000000
max      42901.000000
Name: duration_seconds, dtype: float64


In [17]:
# Check for is_short column and video format distribution
print("\n" + "="*60)
print("VIDEO FORMAT DISTRIBUTION")
print("="*60)

if 'is_short' in df.columns:
    print("\nis_short column values:")
    print(df['is_short'].value_counts())
    print(f"\nShort videos: {(df['is_short'] == True).sum()}")
    print(f"Long videos: {(df['is_short'] == False).sum()}")
    
    # Option to select short or long
    short_videos = df[df['is_short'] == True].copy()
    long_videos = df[df['is_short'] == False].copy()
else:
    print("No is_short column found")
    short_videos = None
    long_videos = None


VIDEO FORMAT DISTRIBUTION

is_short column values:
is_short
False    8483
True     4356
Name: count, dtype: int64

Short videos: 4356
Long videos: 8483


In [18]:
# Select one random SHORT video
print("\n" + "="*60)
print("SELECTING RANDOM SHORT VIDEO")
print("="*60)

if short_videos is not None and len(short_videos) > 0:
    random_short_video = short_videos.sample(n=1).iloc[0]
    short_video_dict = random_short_video.to_dict()
    print(f"\nRandom Short Video Selected:")
    print(f"Title: {random_short_video.get('title', 'N/A')}")
    print(f"Video ID: {random_short_video.get('videoId', 'N/A')}")
    print(f"Duration: {random_short_video.get('duration', 'N/A')}")
    print(f"Channel: {random_short_video.get('channelTitle', 'N/A')}")
    print(f"View Count: {random_short_video.get('viewCount', 'N/A')}")
else:
    print("No short videos found in the dataset")
    random_short_video = None
    short_video_dict = None


SELECTING RANDOM SHORT VIDEO

Random Short Video Selected:
Title: Bellissimiiii #perte #unboxing #blindbox #sushicat
Video ID: a8YKVUCh_Zw
Duration: PT58S
Channel: Smibie Channel
View Count: 82511


In [19]:
# Function to extract ALL links from description
def extract_all_links(text):
    """Extract all URLs from text"""
    if pd.isna(text):
        return []
    
    text = str(text)
    # Pattern for http(s) URLs
    url_pattern = r'https?://[^\s]+'
    # Also look for shortened URLs
    short_pattern = r'(?:youtu\.be|bit\.ly|ow\.ly|tinyurl|goo\.gl)\S+'
    
    urls = re.findall(url_pattern, text)
    urls.extend(re.findall(short_pattern, text))
    
    return list(set(urls))  # Remove duplicates

print("\n" + "="*60)
print("CREATOR'S DESCRIPTION & LINKS")
print("="*60)

if short_video_dict is not None:
    description = short_video_dict.get('description', '')
    tags = short_video_dict.get('tags', '')
    
    print(f"\n📝 FULL DESCRIPTION:")
    print("-" * 60)
    if description:
        print(description)
    else:
        print("[No description provided]")
    
    print(f"\n🏷️  TAGS:")
    print("-" * 60)
    if tags:
        print(tags)
    else:
        print("[No tags]")
    
    # Extract all links
    all_links = extract_all_links(description)
    
    print(f"\n🔗 ALL LINKS FOUND IN DESCRIPTION:")
    print("-" * 60)
    if all_links:
        for i, link in enumerate(all_links, 1):
            print(f"{i}. {link}")
    else:
        print("[No links found]")


CREATOR'S DESCRIPTION & LINKS

📝 FULL DESCRIPTION:
------------------------------------------------------------
nan

🏷️  TAGS:
------------------------------------------------------------
nan

🔗 ALL LINKS FOUND IN DESCRIPTION:
------------------------------------------------------------
[No links found]


In [20]:
# Function to extract video IDs from description
def extract_video_ids_from_description(description):
    """Extract YouTube video IDs from description text"""
    if pd.isna(description):
        return []
    
    description = str(description)
    video_ids = []
    
    # Pattern 1: youtube.com/watch?v=VIDEO_ID
    pattern1 = r'youtube\.com/watch\?v=([a-zA-Z0-9_-]{11})'
    # Pattern 2: youtu.be/VIDEO_ID
    pattern2 = r'youtu\.be/([a-zA-Z0-9_-]{11})'
    
    video_ids.extend(re.findall(pattern1, description))
    video_ids.extend(re.findall(pattern2, description))
    
    return list(set(video_ids))  # Remove duplicates

# Check for potential parent video links
print("\n" + "="*60)
print("ANALYZING POTENTIAL PARENT VIDEO LINKS")
print("="*60)

if short_video_dict is not None:
    description = short_video_dict.get('description', '')
    
    # Extract video IDs from description
    linked_video_ids = extract_video_ids_from_description(description)
    
    if linked_video_ids:
        print(f"\n✅ Found {len(linked_video_ids)} YouTube video link(s) in description:")
        for vid_id in linked_video_ids:
            print(f"\n  Video ID: {vid_id}")
            print(f"  Link: https://www.youtube.com/watch?v={vid_id}")
            # Try to find this video in our dataset
            potential_parent = df[df['videoId'] == vid_id]
            if not potential_parent.empty:
                parent_video = potential_parent.iloc[0]
                print(f"  ✓ FOUND IN DATASET!")
                print(f"    Title: {parent_video.get('title', 'N/A')}")
                print(f"    Duration: {parent_video.get('duration', 'N/A')}")
                print(f"    Is Short: {parent_video.get('is_short', 'N/A')}")
                print(f"    Channel: {parent_video.get('channelTitle', 'N/A')}")
            else:
                print(f"  ⚠️  NOT in current dataset (but exists on YouTube)")
    else:
        print(f"\n❌ No YouTube video links found in description")
        print(f"\nSearching for potential source by channel & publish date...")
        
        # Try to find long videos from same channel published around same time
        channel_id = short_video_dict.get('channelId')
        if channel_id:
            channel_videos = df[df['channelId'] == channel_id].copy()
            long_channel_videos = channel_videos[channel_videos['is_short'] == False]
            
            if len(long_channel_videos) > 0:
                print(f"\nFound {len(long_channel_videos)} long-form videos from same channel ({random_short_video.get('channelTitle')})")
                # Show 3 most recent
                recent = long_channel_videos.sort_values('publishedAt', ascending=False).head(3)
                print(f"\nMost recent long videos from this channel:")
                for idx, (i, video) in enumerate(recent.iterrows()):
                    print(f"  {idx+1}. {video['title'][:60]}... ({video['duration']})")
            else:
                print(f"No long-form videos found from same channel")


ANALYZING POTENTIAL PARENT VIDEO LINKS

❌ No YouTube video links found in description

Searching for potential source by channel & publish date...

Found 820 long-form videos from same channel (Smibie Channel)

Most recent long videos from this channel:
  1. alone in New York City!! 👩🏻‍🎓🇺🇸 | NYC VLOG 09/02/26... (PT40M20S)
  2. un freddo letale tra i grattacieli e Chinatown, ci vivrei? ❄... (PT47M48S)
  3. NY COME NON L'AVETE MAI VISTA!! ☃️🇺🇸 | NYC VLOG 01/02/26... (PT40M38S)


In [21]:
# Display FULL JSON for short video
print("\n" + "="*60)
print("COMPLETE SHORT VIDEO JSON")
print("="*60)
if short_video_dict is not None:
    print(json.dumps(short_video_dict, indent=2, ensure_ascii=False))
else:
    print("No short video to display")


COMPLETE SHORT VIDEO JSON
{
  "videoId": "a8YKVUCh_Zw",
  "publishedAt": "2024-10-31T10:21:31Z",
  "channelId": "UCDoBPEFIUcuMVkQ4I1ql82w",
  "channelTitle": "Smibie Channel",
  "title": "Bellissimiiii #perte #unboxing #blindbox #sushicat",
  "description": NaN,
  "tags": NaN,
  "categoryId": 22,
  "liveBroadcastContent": "none",
  "defaultLanguage": "it",
  "defaultAudioLanguage": "it",
  "localized_title": "Bellissimiiii #perte #unboxing #blindbox #sushicat",
  "localized_description": NaN,
  "thumbnail_default_url": "https://i.ytimg.com/vi/a8YKVUCh_Zw/default.jpg",
  "thumbnail_default_width": 120,
  "thumbnail_default_height": 90,
  "thumbnail_medium_url": "https://i.ytimg.com/vi/a8YKVUCh_Zw/mqdefault.jpg",
  "thumbnail_medium_width": 320,
  "thumbnail_medium_height": 180,
  "thumbnail_high_url": "https://i.ytimg.com/vi/a8YKVUCh_Zw/hqdefault.jpg",
  "thumbnail_high_width": 480,
  "thumbnail_high_height": 360,
  "thumbnail_standard_url": "https://i.ytimg.com/vi/a8YKVUCh_Zw/sddefaul

In [22]:
# Convert selected videos to JSON files and save
output_dir = Path('./youtube_downloads')
output_dir.mkdir(exist_ok=True)

saved_files = []

# Save short video
if short_video_dict is not None:
    short_video_id = short_video_dict.get('videoId', 'short_video')
    short_json_filename = f"short_video_{short_video_id}.json"
    short_json_path = output_dir / short_json_filename
    
    with open(short_json_path, 'w', encoding='utf-8') as f:
        json.dump(short_video_dict, f, indent=2, ensure_ascii=False)
    
    saved_files.append(str(short_json_path))
    print(f"Short video JSON saved!")
    print(f"Path: {short_json_path}")

Short video JSON saved!
Path: youtube_downloads\short_video_a8YKVUCh_Zw.json


In [23]:
# Final Summary
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"\nTotal videos in dataset: {len(df)}")
print(f"Short videos: {len(short_videos) if short_videos is not None else 0}")
print(f"Long videos: {len(long_videos) if long_videos is not None else 0}")
print(f"\nFiles saved: {len(saved_files)}")
for file in saved_files:
    print(f"  - {file}")


FINAL SUMMARY

Total videos in dataset: 12839
Short videos: 4356
Long videos: 8483

Files saved: 1
  - youtube_downloads\short_video_a8YKVUCh_Zw.json
